# Convolutional networks and batch normalization

University of Michigan EECS 498/598, Winter 2022. Compact companion to the completed course assignment.

**Static validation only; not executed.** No ordinary-notebook outputs or scores are copied here. Run cells in order only when execution is intended; training variants retain the ordinary notebook parameters. Existing course dependencies and data are required; no installation or repository cloning is performed. Results are isolated under `core_results/convolutional_networks`.

## Setup

In [ ]:
from pathlib import Path
import os
import sys
import importlib

candidates = [Path.cwd(), *Path.cwd().parents, Path('/content/drive/MyDrive/umich-dlcv-wi2022')]
PROJECT_ROOT = next((p for p in candidates if (p / 'assignments' / 'a3' / 'convolutional_networks_core.ipynb').is_file()), None)
if PROJECT_ROOT is None:
    raise FileNotFoundError('Open the canonical project or assignment directory; mount Drive first in Colab.')
ASSIGNMENT_DIR = PROJECT_ROOT / 'assignments' / 'a3'
if 'eecs598' in sys.modules and Path(sys.modules['eecs598'].__file__).resolve().parent != (ASSIGNMENT_DIR / 'eecs598').resolve():
    raise RuntimeError('Restart the kernel before switching assignments: eecs598 differs across assignments.')
os.chdir(ASSIGNMENT_DIR)
sys.path.insert(0, str(ASSIGNMENT_DIR))
sys.modules.setdefault('imp', importlib)
import torch
import matplotlib.pyplot as plt
import eecs598
from eecs598 import reset_seed

device = 'cuda' if torch.cuda.is_available() else 'cpu'
RESULTS_DIR = ASSIGNMENT_DIR / 'core_results' / 'convolutional_networks'
RESULTS_DIR.mkdir(parents=True, exist_ok=True)
reset_seed(0)
print('Device:', device)


## Data

In [ ]:
DATA_DIR = next((p for p in (ASSIGNMENT_DIR, PROJECT_ROOT / 'assignments' / 'a1') if (p / 'cifar-10-batches-py').is_dir()), None)
if DATA_DIR is None:
    raise FileNotFoundError('Prepare the course CIFAR-10 cache before running this notebook.')
try:
    os.chdir(DATA_DIR)
    data_dict = eecs598.data.preprocess_cifar10(cuda=device == 'cuda', dtype=torch.float64, flatten=False, bias_trick=False, show_examples=False)
finally:
    os.chdir(ASSIGNMENT_DIR)

from eecs598 import Solver
def save_solver_histories(stage, trained_solvers):
    histories = {str(label): {'loss': trained.loss_history, 'train_accuracy': trained.train_acc_history, 'validation_accuracy': trained.val_acc_history} for label, trained in trained_solvers.items()}
    torch.save(histories, RESULTS_DIR / f'{stage}_histories.pt')
    for label, trained in trained_solvers.items():
        plt.plot(trained.val_acc_history, label=str(label))
    plt.xlabel('Epoch')
    plt.ylabel('Validation accuracy')
    plt.legend()
    plt.savefig(RESULTS_DIR / f'{stage}_validation.png', bbox_inches='tight')
    plt.show()


## Overfit a three-layer convolutional model

In [ ]:
from convolutional_networks_core import ThreeLayerConvNet
from fully_connected_networks_core import adam
reset_seed(0)
num_train = 100
small_data = {'X_train': data_dict['X_train'][:num_train], 'y_train': data_dict['y_train'][:num_train], 'X_val': data_dict['X_val'], 'y_val': data_dict['y_val']}
model = ThreeLayerConvNet(weight_scale=0.001, dtype=torch.float32, device=device)
solver = Solver(model, small_data, num_epochs=30, batch_size=50, update_rule=adam, optim_config={'learning_rate': 0.002}, verbose=True, print_every=1, device=device)
solver.train()
save_solver_histories('stage_43', {'model': solver})


## Train the three-layer model on full data

In [ ]:
from convolutional_networks_core import ThreeLayerConvNet
from fully_connected_networks_core import adam
reset_seed(0)
model = ThreeLayerConvNet(weight_scale=0.001, hidden_dim=500, reg=0.001, dtype=torch.float, device=device)
solver = Solver(model, data_dict, num_epochs=1, batch_size=64, update_rule=adam, optim_config={'learning_rate': 0.002}, verbose=True, print_every=50, device=device)
solver.train()
save_solver_histories('stage_47', {'model': solver})


## Overfit a deep convolutional model

In [ ]:
from convolutional_networks_core import DeepConvNet, find_overfit_parameters
from fully_connected_networks_core import adam
reset_seed(0)
num_train = 50
small_data = {'X_train': data_dict['X_train'][:num_train], 'y_train': data_dict['y_train'][:num_train], 'X_val': data_dict['X_val'], 'y_val': data_dict['y_val']}
input_dims = small_data['X_train'].shape[1:]
weight_scale, learning_rate = find_overfit_parameters()
model = DeepConvNet(input_dims=input_dims, num_classes=10, num_filters=[8, 16, 32, 64], max_pools=[0, 1, 2, 3], reg=1e-05, weight_scale=weight_scale, dtype=torch.float32, device=device)
solver = Solver(model, small_data, print_every=10, num_epochs=30, batch_size=10, update_rule=adam, optim_config={'learning_rate': learning_rate}, device=device)
solver.train(return_best_params=False)
plt.plot(solver.loss_history, 'o')
plt.title('Training loss history')
plt.xlabel('Iteration')
plt.ylabel('Training loss')
plt.show()
val_acc = solver.check_accuracy(solver.X_train, solver.y_train, num_samples=solver.num_train_samples)
print(val_acc)
solver.model.save(str(RESULTS_DIR / 'overfit_deepconvnet.pth'))
save_solver_histories('stage_56', {'model': solver})


## Initialization comparison

In [ ]:
from convolutional_networks_core import DeepConvNet
from fully_connected_networks_core import sgd_momentum
reset_seed(0)
num_train = 10000
small_data = {'X_train': data_dict['X_train'][:num_train], 'y_train': data_dict['y_train'][:num_train], 'X_val': data_dict['X_val'], 'y_val': data_dict['y_val']}
input_dims = data_dict['X_train'].shape[1:]
weight_scales = ['kaiming', 0.1, 0.01, 0.001]
solvers = []
for weight_scale in weight_scales:
    print('Solver with weight scale: ', weight_scale)
    model = DeepConvNet(input_dims=input_dims, num_classes=10, num_filters=[8] * 10 + [32] * 10 + [128] * 10, max_pools=[9, 19], weight_scale=weight_scale, reg=1e-05, dtype=torch.float32, device=device)
    solver = Solver(model, small_data, num_epochs=1, batch_size=128, update_rule=sgd_momentum, optim_config={'learning_rate': 0.002}, print_every=20, device=device)
    solver.train()
    solvers.append(solver)
save_solver_histories('stage_61', dict(zip(weight_scales, solvers)))


## One-minute convolutional training budget

In [ ]:
if device != 'cuda':
    raise RuntimeError('The course one-minute experiment requires a GPU; run other stages independently on CPU.')
from convolutional_networks_core import DeepConvNet, create_convolutional_solver_instance
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = True
solver = create_convolutional_solver_instance(data_dict, torch.float32, device)
solver.train(time_limit=60)
torch.backends.cudnn.benchmark = False
solver.model.save(str(RESULTS_DIR / 'one_minute_deepconvnet.pth'))
print('Validation accuracy:', solver.check_accuracy(data_dict['X_val'], data_dict['y_val']))
print('Test accuracy:', solver.check_accuracy(data_dict['X_test'], data_dict['y_test']))
save_solver_histories('stage_64', {'model': solver})


## Batch normalization comparison

In [ ]:
from convolutional_networks_core import DeepConvNet
reset_seed(0)
num_train = 500
small_data = {'X_train': data_dict['X_train'][:num_train], 'y_train': data_dict['y_train'][:num_train], 'X_val': data_dict['X_val'], 'y_val': data_dict['y_val']}
input_dims = data_dict['X_train'].shape[1:]
bn_model = DeepConvNet(input_dims=input_dims, num_classes=10, num_filters=[16, 32, 32, 64, 64], max_pools=[0, 1, 2, 3, 4], weight_scale='kaiming', batchnorm=True, reg=1e-05, dtype=torch.float32, device=device)
model = DeepConvNet(input_dims=input_dims, num_classes=10, num_filters=[16, 32, 32, 64, 64], max_pools=[0, 1, 2, 3, 4], weight_scale='kaiming', batchnorm=False, reg=1e-05, dtype=torch.float32, device=device)
print('Solver with batch norm:')
bn_solver = Solver(bn_model, small_data, num_epochs=10, batch_size=100, update_rule=adam, optim_config={'learning_rate': 0.001}, print_every=20, device=device)
bn_solver.train()
print('\nSolver without batch norm:')
solver = Solver(model, small_data, num_epochs=10, batch_size=100, update_rule=adam, optim_config={'learning_rate': 0.001}, print_every=20, device=device)
solver.train()
save_solver_histories('stage_94', {'without normalization': solver, 'batch normalization': bn_solver})


## Learning-rate comparison with and without normalization

In [ ]:
from convolutional_networks_core import DeepConvNet
from fully_connected_networks_core import sgd_momentum
reset_seed(0)
num_train = 10000
small_data = {'X_train': data_dict['X_train'][:num_train], 'y_train': data_dict['y_train'][:num_train], 'X_val': data_dict['X_val'], 'y_val': data_dict['y_val']}
input_dims = data_dict['X_train'].shape[1:]
num_epochs = 5
lrs = [0.2, 0.1, 0.05]
lrs = [0.005, 0.01, 0.02]
solvers = []
for lr in lrs:
    print('No normalization: learning rate = ', lr)
    model = DeepConvNet(input_dims=input_dims, num_classes=10, num_filters=[8, 8, 8], max_pools=[0, 1, 2], weight_scale='kaiming', batchnorm=False, reg=1e-05, dtype=torch.float32, device=device)
    solver = Solver(model, small_data, num_epochs=num_epochs, batch_size=100, update_rule=sgd_momentum, optim_config={'learning_rate': lr}, verbose=False, device=device)
    solver.train()
    solvers.append(solver)
bn_solvers = []
for lr in lrs:
    print('Normalization: learning rate = ', lr)
    bn_model = DeepConvNet(input_dims=input_dims, num_classes=10, num_filters=[8, 8, 16, 16, 32, 32], max_pools=[1, 3, 5], weight_scale='kaiming', batchnorm=True, reg=1e-05, dtype=torch.float32, device=device)
    bn_solver = Solver(bn_model, small_data, num_epochs=num_epochs, batch_size=128, update_rule=sgd_momentum, optim_config={'learning_rate': lr}, verbose=False, device=device)
    bn_solver.train()
    bn_solvers.append(bn_solver)
save_solver_histories('stage_98', {**{f'without normalization lr={lr}': trained for lr, trained in zip(lrs, solvers)}, **{f'batch normalization lr={lr}': trained for lr, trained in zip(lrs, bn_solvers)}})


## Compare final validation curves

In [ ]:
for label, group in [('without normalization', solvers), ('batch normalization', bn_solvers)]:
    for lr, trained_solver in zip(lrs, group):
        plt.plot(trained_solver.val_acc_history, label=f'{label}, lr={lr}')
plt.xlabel('Epoch')
plt.ylabel('Validation accuracy')
plt.legend()
plt.savefig(RESULTS_DIR / 'normalization_comparison.png', bbox_inches='tight')
plt.show()
